### ライブラリの準備

###モジュールのインポートとGoogleドライブのマウント

In [ ]:
import os
import glob
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import datetime
#from tqdm import tqdm
from tqdm.notebook import tqdm
import pickle
import random
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from PIL import Image
import skimage.transform
from collections import deque
from typing import Sequence, Dict, Tuple, Union

import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence
from torchvision import models
import torchvision.transforms as T
import torchvision.datasets as dataset
from torchvision.transforms import v2

from timm.scheduler import CosineLRScheduler
from transformers import  get_linear_schedule_with_warmup

#from transformers import AutoImageProcessor, AutoModel, AutoProcessor, CLIPVisionModel
from transformers import BertTokenizer, BertModel, CLIPVisionModel, BertForPreTraining

import sys

import util
import levenshtein
from nltk import bleu_score
import ssl
from torch.amp import autocast, GradScaler

In [ ]:
class PositionalEmbedding(nn.Module):
    '''
    位置埋め込み （Positional embedding）
    dim_embedding: 埋込み次元
    max_len      : 入力の最大系列長
    '''
    def __init__(self, dim_embedding: int, max_len: int=2048):
        super().__init__()

        self.pos_emb = nn.Embedding(max_len, dim_embedding)

    '''
    位置エンコーディングの順伝播
    x: 位置エンコーディングを埋め込む対象のテンソル,
       [バッチサイズ, 系列長, 埋め込み次元]
    '''
    def forward(self, x: torch.Tensor):
        seq = x.shape[1]
        positions = torch.arange(start=0, end=seq, step=1, device=x.device).to(torch.long)
        positions = self.pos_emb(positions)[:seq,:]
        
        return positions

### CaptioningTransformer

In [ ]:
class CaptioningTransformer(nn.Module):
    '''
    CaptioningTransformerのコンストラクタ
    dim_embedding  : 埋め込み次元
    dim_feedforward: FNNの中間特徴次元
    num_heads      : マルチヘッドアテンションのヘッド数
    num_layers     : Transformerデコーダ層の数
    vocab_size     : 辞書の次元
    null_index     : NULLのID
    dropout        : ドロップアウト確率
    '''
    def __init__(self, img_size: int, length_max: int, dim_embedding: int,
                  vocab_size: int, tokenizer, dropout: float=0.1, model_id: str=''):
        super().__init__()

        self.mask_token_id = tokenizer.mask_token_id
        self.pad_token_id = tokenizer.pad_token_id
        self.max_idx_en = len( tokenizer )

        #CLIP
        clip_model_id = "openai/clip-vit-large-patch14-336"
        self.clip_model = CLIPVisionModel.from_pretrained(clip_model_id, output_hidden_states = True)
        images = torch.randn( ( 1, 3, img_size, img_size ) )
        memory = self.clip_model( images )
        memory = memory.last_hidden_state
        img_length = memory.size(1)
        clip_dim = memory.size(2)
        self.ln_memory = nn.LayerNorm( dim_embedding )

        self.emb = nn.Embedding( vocab_size, dim_embedding )
        self.pos_emb = PositionalEmbedding( dim_embedding )

        self.dropout = nn.Dropout( dropout )

        self.dc_linear = nn.Linear( clip_dim * 3, dim_embedding )

        # Down Sampling
        #img_length = 577
        #length_max = 84
        stride = img_length // length_max
        #stride = 6
        self.conv1 = nn.Conv1d( dim_embedding, dim_embedding, 1, stride )
        print( "img_length:", img_length )
        print( "text_length_max:", length_max )
        print( "stride:", stride )
        
        self.bert = BertModel.from_pretrained( model_id )

        ## 単語出力分布計算
        self.ln_outputs = nn.LayerNorm( dim_embedding )
        self.linear = nn.Linear(dim_embedding, vocab_size)

        self.dim_embedding = dim_embedding
        self.length_max = length_max

    ''' CaptioningTransformerの順伝播処理
    features: 画像特徴量 [バッチサイズ, 埋め込み次元]
    captions: 正解キャプション [バッチサイズ, 系列長]
    '''
    def forward(self, images: torch.Tensor, captions: torch.Tensor ):

        self.device = images.device

        caption_lengths = torch.ones( ( captions.size(0) ), dtype=torch.long, device = self.device ) * self.length_max
        masked_captions, mask = self.masking( captions, caption_lengths )
        
        memory = self.clip_model( images )
        memory = self.dense_connector( memory )
        memory = self.dropout( memory )
        memory = self.ln_memory( memory )

        memory = self.conv1( memory.transpose(1,2) ).transpose(1,2)
        
        emb_caption = self.emb( masked_captions ) * math.sqrt(self.dim_embedding)
        emb_caption += self.pos_emb( emb_caption )

        bert_in = torch.cat( [memory, emb_caption], dim = 1 )
        #bert_in_padding_masks = None
        bert_in_padding_masks = torch.ones_like( masked_captions, device = self.device, dtype=torch.float )
        bert_in_padding_masks = torch.cat( [torch.ones( memory.shape[:2], device=model.device ), bert_in_padding_masks], dim = 1 )
        
        outputs = self.bert( inputs_embeds = bert_in, attention_mask = bert_in_padding_masks ).last_hidden_state
        outputs = outputs[:,memory.size(1):,:]
        outputs = self.ln_outputs( outputs )
        logits = self.linear( outputs )
        
        return logits, mask

    def dense_connector(self, memory ):
        tmp1 = torch.tensor([], device = self.device )
        tmp2 = torch.tensor([], device = self.device )
        tmp_full = len( memory.hidden_states )
        tmp_half = tmp_full // 2
        for i in range( 0, tmp_half ):
            tmp1 = torch.cat( [tmp1, memory.hidden_states[i][None]], dim = 0 )
        tmp1 = torch.sum(tmp1, dim=0) / tmp_half
        for i in range( tmp_half, tmp_full ):
            tmp2 = torch.cat( [tmp2, memory.hidden_states[i][None]], dim = 0 )
        tmp2 = torch.sum(tmp2, dim=0 ) / ( tmp_full - tmp_half )
        tmp3 = torch.cat([tmp1, tmp2], dim=-1)
        tmp3 = torch.cat( [ memory.last_hidden_state, tmp3], dim = -1 )
        tmp3 = self.dc_linear( tmp3 )
        return tmp3

    def masking(self, input_x: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor]:

        output = input_x.clone()

        masks = torch.zeros_like( output, device=output.device, dtype=torch.bool )       
        
        #sum_num_mask = 0
        #sum_num_arbi = 0
        #sum_num_nochange = 0
        for n in range( output.size(0) ):
            #all_prob = torch.normal( torch.tensor( 0.7 ), torch.tensor( 0.2 ) )
            all_prob = torch.normal( torch.tensor( 0.8 ), torch.tensor( 0.2 ) )
            all_prob = torch.clamp( all_prob, min = 0.0, max = 1.0 )
            if all_prob > 0.99:
                num_mask = lengths[n]
                num_arbi = 0
                num_nochange = 0
            else:
                #mask_prob0 = torch.normal( torch.tensor( 0.7 ), torch.tensor( 0.2 ) )
                mask_prob0 = torch.normal( torch.tensor( 0.8 ), torch.tensor( 0.2 ) )
                mask_prob0 = torch.clamp( mask_prob0, min = 0.0, max = 1.0 )
                mask_prob = all_prob * mask_prob0
                resi_prob = all_prob * ( 1.0 - mask_prob0 )
                arbi_prob = all_prob * ( resi_prob * 0.5 )
                nochange_prob = all_prob * ( resi_prob * 0.5 )
                num_mask = math.floor( lengths[n].item() * mask_prob )
                num_arbi = math.floor( lengths[n].item() * arbi_prob )
                num_nochange = math.floor( lengths[n].item() * nochange_prob )

            #sum_num_mask += num_mask
            #sum_num_arbi += num_arbi
            #sum_num_nochange += num_nochange
            
            mask_mask = list( random.sample( list(range( 0, lengths[n])),  num_mask ))
            output[n,mask_mask] = self.mask_token_id
            not_mask_mask = [ n for n in range( lengths[n] ) if n not in mask_mask ]
            mask_arbi = random.sample( not_mask_mask, num_arbi )
            for i in range( lengths[n] ):
                if i in mask_arbi:
                    output[n,i] = torch.randint( 0, self.max_idx_en, size=(1,))
            not_mask_arbi = [ n for n in not_mask_mask if n not in mask_arbi ]
            mask_nochange = random.sample( not_mask_arbi, num_nochange )
            not_mask_nochange = [ n for n in not_mask_arbi if n not in mask_nochange ]
            mask = [ False if n in not_mask_nochange else True for n in range(lengths[n]) ]
            masks[n,:lengths[n]] = torch.tensor( mask )

        #print( "sum_num_mask:", sum_num_mask )
        #print( "calculate num mask:", torch.sum( torch.eq( output, self.mask_token_id ).int() ) )
        #print( "sum_num_mask + sum_num_arbi :", sum_num_mask + sum_num_arbi )
        #print( "num not equal:", torch.sum( torch.ne( input_x, output ).int() ) )
        #print( "sum_num_mask + sum_num_arbi + sum_nochange:", sum_num_mask + sum_num_arbi + sum_num_nochange )
        #print( "num of mask True:", torch.sum( torch.eq( masks, True ) ) )
        
        return output, masks

    def my_decode(self, token_list, tokenizer ):

        def my_index( l, x ):
            if x in l:
                return l.index(x)
            else:
                return -1
        if my_index( token_list, tokenizer.sep_token_id ) != -1:
            token_list = token_list[:my_index( token_list, tokenizer.sep_token_id )]
        else:
            token_list = token_list
            
        text = tokenizer.decode( token_list, skip_special_tokens = True )
        
        return text

In [ ]:
class MyDataset(Dataset):
    def __init__(self, file_path: str, img_directory: str, transforms, tokenizer, length_max = None ) -> None:
        super().__init__()
        self.img_directory = img_directory
        self.transforms = transforms
        # TODO: fix to original data
        #画像の前処理
        self.img_file = []
        self.tokens = []
        if length_max == None:
            self.length_max = 0
        else:
            self.length_max = length_max
        length_sum = 0
        with open( file_path, "r" ) as f:
            #line = f.readline()
            #i = 0
            #while line:
            for i, line in enumerate( f ):
                if i % 100000 == 0:
                #    #print( line.split("\t")[0])
                #    #print( line.split("\t")[1])
                    print( "i:", i )
                #i += 1
                self.img_file.append(line.split("\t" )[0])
                caption = line.split("\t")[1].replace( "\r\n", "" ).replace( "\n", "").replace( "\r", "" )
                #print( "caption:", caption )
                id_tokens = tokenizer.encode( caption )
                length_sum += len( id_tokens )
                if length_max == None:
                    if self.length_max < len( id_tokens ):
                        self.length_max = len( id_tokens )
                    #id_tokens = torch.tensor( id_tokens, requires_grad = False  )
                    id_tokens = torch.tensor( id_tokens  )
                else:
                    #id_tokens = torch.tensor( id_tokens, requires_grad = False)[:length_max]
                    id_tokens = torch.tensor( id_tokens )[:length_max]
                
                #print( "id_tokens:", id_tokens )
                self.tokens.append( id_tokens )

                #line = f.readline()
        print("avg len:", length_sum / len( self.tokens ) )    
    
    # ここで取り出すデータを指定している
    def __getitem__(
        self,
        index: int
    ):
        tokens = self.tokens[index]
        img_file = self.img_file[index] + ".jpg"
        img_path = os.path.join( self.img_directory, img_file ) #index番目の画像のパスを取得
        img = Image.open(img_path) #PIL形式で画像を読み込み
        if img.mode != 'RGB':
            img = img.convert("RGB")
        img = self.transforms(img)
        
        return img, tokens

    # この method がないと DataLoader を呼び出す際にエラーを吐かれる
    def __len__(self) -> int:
        return len(self.tokens)

    def length_max(self):
        return self.length_max

In [ ]:
def collate_func(batch: Sequence[Tuple[Union[torch.Tensor, str]]], pad_index, length_max ):
    imgs, tokens = zip(*batch)

    #max_length = 0
    #for target in tokens:
    #    if max_length < len( target ):
    #        max_length = len( target )
    max_length = length_max
    
    targets = []
    lengths = []
    for target in tokens:
        pad_len = max_length - len( target ) 
        input2= F.pad( target, (0, pad_len), mode='constant', value = pad_index)
        targets.append( input2 )
        lengths.append( len( target ) )
    
    imgs = torch.stack( imgs, dim = 0 )
    targets = torch.stack( targets, dim = 0 )
    lengths = torch.tensor( lengths  )
   
    return imgs, targets, lengths

In [ ]:
# 画像のtransformsを定義
transforms = v2.Compose([
    v2.Resize((336, 336)),
    #v2.AutoAugment(),
    #v2.ToTensor(),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    ## Coco データセット 2017 train の平均と標準偏差
    #v2.Normalize((0.456,0.427,0.401),(0.224,0.219,0.231) )
    ## ImageNetデータセットの平均と標準偏差
    #v2.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
    # clip の preprocessor_config.json の平均と標準偏差
    v2.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711))
])

model_id = "google-bert/bert-large-uncased"
tokenizer = BertTokenizer.from_pretrained(model_id)
length_max = 84

# v7 データセット
train_dataset = MyDataset( file_path="../CLIP_LLM_AR/dataset.txt",
                           img_directory = "/mnt/ssd2/v7/img",
                           #img_directory = "smb://192.168.1.2/img/v7/",
                           transforms=transforms, tokenizer = tokenizer, length_max = length_max )

# Subset samplerの生成
test_set, val_set, train_set = util.generate_subset_test_val_train(
    train_dataset, 0.1, 0.1 )
    
# 学習時にランダムにサンプルするためのサンプラー
train_sampler = SubsetRandomSampler(train_set)

# DataLoaderを生成
collate_func_lambda = lambda x: collate_func(x, tokenizer.pad_token_id, length_max )

test_loader = torch.utils.data.DataLoader(
                    train_dataset,
                    #batch_size=config.batch_size,
                    batch_size=1,
                    num_workers=0,
                    sampler=test_set,
                    collate_fn=collate_func_lambda)


###学習におけるハイパーパラメータやオプションの設定

In [ ]:
#device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")
## 辞書（単語→単語ID）の読み込み
#with open('../PreTrain_Decoder/translateDatasetNTT_blank4_pad0/word_to_id2.pkl', 'rb') as f:
#    word_to_id = pickle.load(f)
#max_idx_en = len( word_to_id )
#word_to_id['<mask>'] = max_idx_en
#mask_value = word_to_id['<mask>']
#start_idx = word_to_id['<start>']
#bert_model_path = 'models--google-bert--bert-large-uncased/snapshots/6da4b6a26a1877e173fca3225479512db81a5e5b'
#tokenizer = BertTokenizer.from_pretrained(pretrained_model_name_or_path = bert_model_path )
model_id = "google-bert/bert-large-uncased"
tokenizer = BertTokenizer.from_pretrained(model_id)
model = CaptioningTransformer(img_size = 336, length_max = 84, dim_embedding=1024, vocab_size=len(tokenizer),
                 tokenizer=tokenizer, dropout=0.1, model_id =model_id).to(device)

PATH = "model/model_bert_mask_curr.pth"
if os.path.isfile(PATH):
    checkpoint = torch.load(PATH, map_location=torch.device('cpu'))
    model.load_state_dict(checkpoint['model_state_dict'])
    print( "load parameters." )

images = torch.randn( ( 1, 3, 336,336 ), device = device )
captions = torch.randint( 0, len(tokenizer), size= (1, 84 ), device= device )
outputs, masks = model( images, captions )

print( outputs.size() )
print( masks.size() )

### 推論関数の定義

In [ ]:
# 推論モジュール
@torch.no_grad()
def inference( images, tokenizer, length_max):

    model.eval()
    device = images.device

    memory = model.clip_model( images )
    memory = model.dense_connector( memory )
    memory = model.dropout( memory )
    memory = model.ln_memory( memory )
    memory = model.conv1( memory.transpose(1,2) ).transpose(1,2)

    masked_captions = torch.ones( (memory.size(0), length_max ), dtype=torch.long, device=device ) * tokenizer.mask_token_id
    
    emb_caption = model.emb( masked_captions ) * math.sqrt(model.dim_embedding)
    emb_caption += model.pos_emb( emb_caption )
      
    bert_in = torch.cat( [memory, emb_caption], dim = 1 )
    #bert_in_padding_masks = None
    bert_in_padding_masks = torch.ones_like( masked_captions, device = model.device, dtype=torch.float )
    bert_in_padding_masks = torch.cat( [torch.ones( memory.shape[:2], device=model.device ), bert_in_padding_masks], dim = 1 )
    
    iter_max = 10
    for i in range( iter_max ):
        outputs = model.bert( inputs_embeds = bert_in, attention_mask = bert_in_padding_masks ).last_hidden_state
        outputs = outputs[:,memory.size(1):,:]
        outputs = model.ln_outputs( outputs )
        logits = model.linear( outputs )
        probabilities = torch.nn.functional.softmax( logits, dim = 2 )
        captions = torch.argmax( logits, dim = 2 )

        if i < iter_max - 1:
            masked_captions = []
            for n in range( outputs.size(0) ):
                max_prob = torch.max( probabilities[n,:,:], dim = 1 ).values
                sorted_max_prob = torch.sort( max_prob, dim = 0 ).values
                masked_caption = captions[n]
                #num_mask = torch.sum( torch.eq( masked_caption, tokenizer.mask_token_id ).int() )
                kosuu_mask = math.floor(( iter_max - i - 1 ) * length_max / iter_max ) 
                #if kosuu_mask  - 1 < 0:
                #    kosuu_mask = 1
                #if num_mask > kosuu_mask:
                #    kosuu_mask = num_mask
                thresh = sorted_max_prob[ kosuu_mask - 1 ]
                t_indices = max_prob < thresh
                masked_caption[t_indices] = tokenizer.mask_token_id
                masked_caption[0] = tokenizer.cls_token_id
                masked_captions.append( masked_caption )

            masked_captions = torch.stack( masked_captions, dim = 0 )
            emb_caption = model.emb( masked_captions ) * math.sqrt(model.dim_embedding)
            emb_caption += model.pos_emb( emb_caption )
            bert_in = torch.cat( [memory, emb_caption], dim = 1 )
            #bert_in_padding_masks = None
            bert_in_padding_masks = torch.ones_like( masked_captions, device = model.device )
            bert_in_padding_masks = torch.cat( [torch.ones( memory.shape[:2], device=model.device ), bert_in_padding_masks], dim = 1 )

        
    return captions

In [ ]:
# 推論モジュール
@torch.no_grad()
def inference2( images, tokenizer, length_max , top_k = None, temperature = 0.0):

    model.eval()
    device = images.device

    memory = model.clip_model( images )
    memory = model.dense_connector( memory )
    memory = model.dropout( memory )
    memory = model.ln_memory( memory )
    memory = model.conv1( memory.transpose(1,2) ).transpose(1,2)

    masked_captions = torch.ones( (memory.size(0), length_max ), dtype=torch.long, device=device ) * tokenizer.mask_token_id
    
    emb_caption = model.emb( masked_captions ) * math.sqrt(model.dim_embedding)
    emb_caption += model.pos_emb( emb_caption )
      
    bert_in = torch.cat( [memory, emb_caption], dim = 1 )
    #bert_in_padding_masks = None
    bert_in_padding_masks = torch.ones_like( masked_captions, device = model.device, dtype=torch.float )
    bert_in_padding_masks = torch.cat( [torch.ones( memory.shape[:2], device=model.device ), bert_in_padding_masks], dim = 1 )
    
    iter_max = 10
    for i in range( iter_max ):
        outputs = model.bert( inputs_embeds = bert_in, attention_mask = bert_in_padding_masks ).last_hidden_state
        outputs = outputs[:,memory.size(1):,:]
        outputs = model.ln_outputs( outputs )
        logits = model.linear( outputs )
        # New: Filter logits with top_k sampling
        if top_k is not None:
            # Keep only top_k values
            top_logits, _ = torch.topk(logits, top_k, dim = 2) # [b, seq_len, k ]
            min_val = top_logits[:,:, -1][:,:,None] #[b, seq_len, 1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)            

        # New: Apply temperature scaling
        if temperature > 0.0:
            logits = logits / temperature

            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len, k)
            probs2 = probs.view( probs.size(0) * probs.size(1), probs.size(2))
            captions2 = torch.multinomial(probs2, num_samples=1)  # (batch_size * seq_len, 1)
            captions = captions2.view( probs.size(0), probs.size(1) )
            adapted_probs2 = torch.tensor( [ probs2[n,captions2[n]] for n in range( captions2.size(0) ) ] )
            adapted_probs = adapted_probs2.view( probs.size(0), probs.size(1) )
        
            ## Sample from the distribution
            #idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        ## Otherwise same as before: get idx of the vocab entry with the highest logits value
        else:
            #idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)
            probs = torch.softmax(logits, dim=-1)
            captions = torch.argmax( logits, dim = -1 )

        probabilities = probs
        
        if i < iter_max - 1:
            masked_captions = []
            for n in range( outputs.size(0) ):
                #max_prob = torch.max( probabilities[n,:,:], dim = 1 ).values
                adapted_prob = adapted_probs[n]
                sorted_adapted_prob = torch.sort( adapted_prob, dim = 0 ).values
                masked_caption = captions[n]
                kosuu_mask = math.floor(( iter_max - i - 1 ) * length_max / iter_max ) 
                thresh = sorted_adapted_prob[ kosuu_mask - 1 ]
                t_indices = adapted_prob < thresh
                masked_caption[t_indices] = tokenizer.mask_token_id
                masked_caption[0] = tokenizer.cls_token_id
                masked_captions.append( masked_caption )

            masked_captions = torch.stack( masked_captions, dim = 0 )
            emb_caption = model.emb( masked_captions ) * math.sqrt(model.dim_embedding)
            emb_caption += model.pos_emb( emb_caption )
            bert_in = torch.cat( [memory, emb_caption], dim = 1 )
            #bert_in_padding_masks = None
            bert_in_padding_masks = torch.ones_like( masked_captions, device = model.device )
            bert_in_padding_masks = torch.cat( [torch.ones( memory.shape[:2], device=model.device ), bert_in_padding_masks], dim = 1 )

    return captions

### テスト

In [ ]:
test_num = 21
## top_k = None だと inference1 
top_k = None
#top_k = 5
temperature = 0.5

my_decode = False
#my_decode = True

# Subset samplerの生成
test_set, val_set, train_set = util.generate_subset_test_val_train(
    train_dataset, 0.1, 0.1 )

test_set = test_set[:test_num]

test_loader = torch.utils.data.DataLoader(
                    train_dataset,
                    #batch_size=config.batch_size,
                    batch_size=1,
                    num_workers=0,
                    sampler=test_set,
                    collate_fn=collate_func_lambda)

test_pr_coef = 1

fn = bleu_score.SmoothingFunction().method7

transforms_inv = v2.Compose([
    v2.Normalize((-0.48145466/0.26862954, -0.4578275/0.26130258, -0.40821073/0.27577711), (1/0.26862954,1/0.26130258,1/0.27577711)),
    v2.ToPILImage()
])

# test
with tqdm(test_loader) as pbar:
    pbar.set_description(f'[テスト]')
    if top_k is not None:
        print(f'top_k: {top_k}' )
        print(f'temperature: {temperature}' )

    # 評価モード
    model.eval()

    test_errors = deque()
    test_bleus = deque()
    n_iter = 0
    length_max = 84
    for k, (imgs, captions, caption_lengths) in enumerate( pbar ):
        #if k > 20:
        #    break
        # ミニバッチを設定
        imgs = imgs.to(device)
        captions = captions.to(device)
        #caption_lengths = torch.tensor( caption_lengths ).to(config.device)
        
        with torch.no_grad():
            #prop_logits = inference(imgs, tokenizer, length_max )
            #hypo_ids = torch.argmax( prop_logits, dim = 2 )
            if top_k is None:
                hypo_ids = inference(imgs, tokenizer, length_max )
            else:
                hypo_ids = inference2(imgs, tokenizer, length_max, top_k, temperature )
        
        n = 0
        hypo_sentence = []
        ref_sentence = []
        ref_imgs = []
        total_error = 0
        total_token_length = 0
        total_bleu = 0
        for (hypo_id, caption, img ) in zip( hypo_ids, captions, imgs ):
            if my_decode == True:
                hypo = model.my_decode( hypo_id.tolist(), tokenizer )
                hypo_tokens = tokenizer.tokenize( hypo )
                reference = model.my_decode( caption.tolist(), tokenizer )
                ref_tokens = tokenizer.tokenize( reference )
            else:
                hypo = tokenizer.decode( hypo_id.tolist(), skip_special_tokens = True )
                hypo_tokens = tokenizer.tokenize( hypo )
                reference = tokenizer.decode( caption.tolist(), skip_special_tokens = True )
                ref_tokens = tokenizer.tokenize( reference )
            
            # 認識誤りを計算
            (error, substitute, delete, insert, ref_length) = levenshtein.calculate_error(hypo_tokens,ref_tokens)
            
            # 誤り文字数を累積する
            total_error += error
            # 文字の総数を累積する
            total_token_length += ref_length

            bleu = bleu_score.sentence_bleu( [reference], hypo, smoothing_function=fn  )
        
            total_bleu += bleu

            inv_img = transforms_inv( img )
            plt.imshow( inv_img )
            plt.axis('off')
            plt.show()
            print( "hypo:", hypo )
            print( "refe:", reference )
            print( "this pic. WER :", error / ref_length )
            print( "this pic. BLEU:", bleu )
            test_errors.append(error / ref_length)
            test_bleus.append(bleu)
            n_iter += 1
            print(f'test number = {n_iter} average, WER = {torch.Tensor(test_errors).mean().item()}, BLEU = {torch.Tensor(test_bleus).mean().item()}')
            print( "\n\n" )
            
                
            #if len(test_errors) > config.moving_avg:
            if len(test_errors) > 100:
                test_errors.popleft()
                test_bleus.popleft()
            pbar.set_postfix({
                #'loss': torch.Tensor(test_losses).mean().item(),
                'WER': torch.Tensor(test_errors).mean().item(),
                'BLEU': torch.Tensor(test_bleus).mean().item()
            })                

# 表示
test_error = np.mean( test_errors )
test_bleu = np.mean( test_bleus )
if top_k is not None:
    print(f'top_k: {top_k}' )
    print(f'temperature: {temperature}' )
print(f'test {n_iter} average WER : {test_error}')
print(f'test {n_iter} average BLEU: {test_bleu}')

In [ ]:
reference="A selection of scissors, including pinking shears, under a glass shelf."
ref = reference.split( " " )
hypo="A bunch of scissors that are on a table.OTHER.R."
hyp = hypo.split( " " )

(error, substitute, delete, insert, ref_length) = levenshtein.calculate_error(hyp,ref)

print( error, substitute, delete, insert, ref_length )
print( error / ref_length * 100 )